# A magyar gyorsforgalmi úthálózat koncessziós rendszere
## Reprodukálható pénzügyi-mérnöki audit és számítási elemzés (Open science)

Ez a Jupyter Notebook a **„A magyar gyorsforgalmi úthálózat koncessziós rendszerének komplex elemzése”** című tanulmány összes számítását, adatmodelljét és érzékenységi vizsgálatát tartalmazza teljesen reprodukálható formában.

### Primer adatforrások
1. **Állami Számvevőszék (ÁSZ):** 1118. számú jelentés az M6/M60 autópálya PPP beruházásairól (alapdíj: 22,3 M Ft/km/hó, hiányossági kötbérlevonások).
2. **Központi Statisztikai Hivatal (KSH):**
   - STADAT 1.1.1.2: Fogyasztóiár-indexek (CPI).
   - STADAT 1.1.1.31: Az építőipar termelői árindexei (Mélyépítés / Egyéb építmény alágazat).
   - STADAT 1.1.1.32: Építményfajták termelői árindexei (Utak, autópályák építése alcsoport).
3. **Kúria és Bírósági Ítéletek:**
   - Kúria Pfv.IV.21.194/2023/15. ítélet (közérdekű adatok kiadása a koncessziós eljárásban).
   - Alkotmánybíróság 3372/2024. (X. 8.) AB végzés.
4. **Magyar Közút Nonprofit Zrt.:** 2021. évi auditált éves beszámoló (386,1 Mrd Ft bevételi/támogatási bázis).
5. **Európai PPP Szakértői Központ (EPEC / EIB):** The Guide to Guidance – Value for Money Assessment (WACC módszertan).


In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Projekt gyökérkönyvtárának hozzáadása az importáláshoz
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.ksh_chains import get_rebased_ksh_summary, load_cpi_annual
from src.km_tariff_comparison import (
    calculate_m6_base_annual,
    method1_domestic_construction,
    method2_contractual_fx,
    calculate_relative_deltas,
    get_full_tariff_comparison_table
)
from src.npv_model import (
    fisher_nominal_rate,
    annuity_factor,
    calculate_real_npv_sensitivity,
    calculate_magyar_kozut_illustrative,
    get_npv_component_breakdown
)
from src.wacc_spread import (
    calculate_wacc,
    calculate_financing_spread,
    get_wacc_sensitivity_matrix
)
from src.engineering_m1 import (
    calculate_m1_widening_unit_costs,
    get_engineering_benchmark_table
)
from src.penalties_sla import get_sla_comparison_table, evaluate_modern_sla

# Matplotlib beállítások
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.figsize'] = (10, 5)
print("Modulok és adatok sikeresen betöltve!")


---
## 1. KSH építőipari és útépítési láncszorzatok (2010 -> 2024)

A 2010 előtti PPP díjak reálértékre történő átszámításának elsődleges alapja a hivatalos KSH termelői árindexek éves láncszorzata:
$$I_{2010\to2024} = \prod_{t=2011}^{2024} \frac{\text{Index}_t}{100}$$


In [ ]:
ksh_summary = get_rebased_ksh_summary()
ksh_summary[["Kategória", "2010 bázis", "2015 szint", "2020 szint", "2024 kumulált szorzó", "Áremelkedés (%)"]]


---
## 2. Fajlagos kilométerdíjak összehasonlítása (M6 PPP vs. MKIF 2022)

Az ÁSZ 1118. jelentés szerinti 2010-es havi 22,3 M Ft/km/hó alapdíj évesítve **267,6 M Ft/km/év**.
Két összehasonlító módszert alkalmazunk:
1. **KSH belföldi építőipari termelői indexálás:** Mélyépítés esetén 628,3 M Ft/km/év, Utak alcsoport esetén 652,4 M Ft/km/év.
2. **Szerződéses EUR deviza + EU infláció:** 971 700 EUR * 1,45 * 395 Ft/EUR = 556,5 M Ft/km/év.
3. **MKIF 2022 indikatív induló átlag:** 525,0 M Ft/km/év.


In [ ]:
tariff_df = get_full_tariff_comparison_table()
display(tariff_df)

# Vizuális összehasonlító grafikon
categories = [
    'M6 Bázis (2010 nom.)',
    'MKIF Indikatív (2022)',
    'M6 Reál (EUR deviza)',
    'M6 Reál (KSH Mélyépítés)',
    'M6 Reál (KSH Utak)'
]
values = [267.6, 525.0, 556.5, 628.3, 652.4]
colors = ['#95a5a6', '#2ecc71', '#3498db', '#e67e22', '#e74c3c']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(categories, values, color=colors, height=0.55)
ax.set_xlabel('Fajlagos Díj (Millió Ft / km / év)', fontsize=11, fontweight='bold')
ax.set_title('Fajlagos Kilométerdíjak Összevetése (M6 PPP vs. MKIF Koncesszió)', fontsize=13, fontweight='bold', pad=15)
ax.axvline(525.0, color='#27ae60', linestyle='--', linewidth=1.5, alpha=0.7, label='MKIF 525 M Ft referencia')

for bar in bars:
    w = bar.get_width()
    ax.text(w + 8, bar.get_y() + bar.get_height()/2, f'{w:.1f} M Ft', ha='left', va='center', fontweight='bold')

ax.set_xlim(0, 750)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


---
## 3. Diszkontálás, Fisher-egyenlet és jelenérték (NPV) érzékenység

A Fisher-összefüggés alapján:
$$1 + r_n = (1 + r_r)(1 + \pi)$$
Ha a pénzáramok 100%-ban CPI-követők, az infláció és a nominális diszkontráta inflációs prémiuma semlegesíti egymást, és a modell visszavezethető a bázisáras reálérték annuitására:
$$\text{PV} = \text{RÁD}_0 \times A_{35, r_r}, \quad \text{ahol} \quad A_{35, r_r} = \frac{1 - (1 + r_r)^{-35}}{r_r}$$


In [ ]:
sens_df = calculate_real_npv_sensitivity(rad0_mrd_huf=375.0)
display(sens_df[["Reál diszkontráta (r_r)", "Annuitási szorzó (A_35)", "Reál jelenérték (NPV)"]])

# Folytonos jelenérték-görbe rajzolása r_r függvényében
rr_range = np.linspace(0.01, 0.06, 100)
npv_curve = [375.0 * annuity_factor(r, 35) for r in rr_range]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rr_range * 100, npv_curve, color='#2980b9', linewidth=2.5, label='Reál NPV (RAD_0 = 375 Mrd Ft)')
ax.axvline(3.5, color='#e74c3c', linestyle=':', linewidth=1.5, label='Bázis diszkontráta: r_r = 3.5% (7 500 Mrd Ft)')
ax.scatter([3.5], [7500], color='#e74c3c', s=80, zorder=5)

ax.set_xlabel('Reál diszkontráta (%)', fontsize=11, fontweight='bold')
ax.set_ylabel('Jelenérték (Milliárd Ft)', fontsize=11, fontweight='bold')
ax.set_title('35 Éves Koncessziós Jelenérték Érzékenysége a Reálkamatra', fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()


---
## 4. Magyar Közút szemléltető példa és jelenérték-dekonstrukció

A több évtizedes nominális pénzáramok puszta összeadásának torzító hatását szemlélteti a Magyar Közút 2021-es költségvetésének (386 Mrd Ft) 35 éves felfuttatása:
$$\sum_{t=1}^{35} 386 \times 1,035^t \approx \mathbf{26\ 640 \text{ milliárd Ft}}$$


In [ ]:
mk = calculate_magyar_kozut_illustrative()
print(f"Magyar Közút 35 éves szemléltető felfuttatás (3.5% infláció):")
print(f"  - Év végi konvencióval:  {mk['end_of_year_mrd_huf']:,} Mrd Ft".replace(",", " "))
print(f"  - Év eleji konvencióval: {mk['start_of_year_mrd_huf']:,} Mrd Ft\n".replace(",", " "))

comp_df = get_npv_component_breakdown()
display(comp_df[["Főcsoport", "Pénzáram-komponens", "Időtáv", "Becsült NPV (Mrd Ft)"]])


---
## 5. Súlyozott átlagos tőkeköltség (WACC) és szuverén finanszírozási spread

Az EPEC módszertan szerinti WACC:
$$WACC = \left( \frac{E}{V} \times r_e \right) + \left( \frac{D}{V} \times r_d \times (1 - T_c) \right)$$
Benchmark paraméterekkel:
$$WACC = (0,15 \times 0,12) + (0,85 \times 0,055 \times 0,91) = \mathbf{6,05\%}$$
Szuverén hozam (ÁKK 2021): $r_g = 2,85\%$.
Finanszírozási különbözet (Spread): $\mathbf{+3,20\%}$ (+320 bázispont).


In [ ]:
wacc_res = calculate_wacc()
spread_res = calculate_financing_spread(wacc_res["wacc"])
print(f"Magán WACC: {wacc_res['wacc_pct']}%")
print(f"ÁKK referenciahozam: {spread_res['sovereign_yield_pct']}%")
print(f"Finanszírozási spread: +{spread_res['spread_pct']}% ({int(spread_res['spread_bps'])} bázispont)")

wacc_matrix = get_wacc_sensitivity_matrix()
display(wacc_matrix[["Saját tőke elvárt hozam (r_e)", "Hitelkamatláb (r_d)", "WACC (%)", "Finanszírozási Spread"]])


---
## 6. M1 forgalom alatti kapacitásbővítés és zöldmezős mérnöki költségek

Az M1 78 km-es (M0 - Győr) forgalom alatti 2x3 sávos szélesítésének fajlagos költségmodellje:
- Közvetlen pályaszerkezet: 4,0 – 5,5 Mrd Ft/km.
- All-in teljes beruházási keret (620 – 800 Mrd Ft): 7,9 – 10,3 Mrd Ft/km.
- Zöldmezős referenciák: Síkvidéki 3,8 – 4,8 Mrd Ft/km, hegyvidéki 5,2 – 5,8 Mrd Ft/km.


In [ ]:
eng_table = get_engineering_benchmark_table()
display(eng_table[["Mérnöki kategória / Típus", "Fajlagos költség (Mrd Ft / km)", "Definíciós megjegyzés"]])


---
## 7. Szolgáltatási szintek és kötbérrendszer (ÁSZ 1118 vs. MKIF 2022)

- **2010 előtti PPP (ÁSZ 1118):** $RÁD_{\text{tényleges}} = RÁD_{\text{bázis}} \times K_f \times K_h - D_{\text{egyéb}}$ (az ÁSZ által vizsgált 10 hónapból 6-ban volt tényleges levonás).
- **2022-es Koncesszió (MKIF):** Folyamatosan mozgó lézeres mérőautók, határértékek:
  - $\text{IRI} \le 1,8$ mm/m
  - $\text{Nyomvályú} \le 8,0$ mm
  - $\text{SFC} \ge 0,45$


In [ ]:
sla_table = get_sla_comparison_table()
display(sla_table)

# Demonstrációs lézeres SLA kiértékelés
test_section = evaluate_modern_sla(measured_iri=1.45, measured_rut_depth_mm=5.8, measured_sfc=0.52)
print("Példa szakasz minősítése (IRI=1.45, Nyomvályú=5.8mm, SFC=0.52):")
print(f"Megfelelő minőség: {test_section['compliant']} (Kötbér tüzelés: {test_section['penalty_triggered']})")
